# QNN Stack Corrosion — Raw IQ + Q-NAS + PQ-FiLM
**Classes:** Stack 1.0g · 1.5g · 2.0g · 2.5g (4-class classification + gram regression)

In [1]:
import os, sys, time, json, re, warnings, random, copy
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from scipy.signal import get_window
from scipy.stats import skew, kurtosis as scipy_kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, r2_score, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
def set_seed(s):
    os.environ['PYTHONHASHSEED'] = str(s)
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


/home/sammarv/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Dataset config ────────────────────────────────────────────────────────────
BASE_DIR = Path('/home/sammarv/quantum_corrosion')

TARGET_CLASSES = [
    {'id': 5, 'name': 'Stack 1.0g', 'type': 'stack', 'path_sub': 'stack/1/1gSTACK',   'gram': 1.0},
    {'id': 6, 'name': 'Stack 1.5g', 'type': 'stack', 'path_sub': 'stack/1.5/1.5gSTACK', 'gram': 1.5},
    {'id': 7, 'name': 'Stack 2.0g', 'type': 'stack', 'path_sub': 'stack/2/2gSTACK',   'gram': 2.0},
    {'id': 8, 'name': 'Stack 2.5g', 'type': 'stack', 'path_sub': 'stack/2.5/2.5gSTACK', 'gram': 2.5},
]

# Internal label = index into TARGET_CLASSES
N_CLASSES   = len(TARGET_CLASSES)
GRAM_VALUES = {i: tc['gram'] for i, tc in enumerate(TARGET_CLASSES)}
LABEL_NAMES = [tc['name'] for tc in TARGET_CLASSES]

OUT_DIR = BASE_DIR / 'results/stack_qnn'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Signal config (identical to corrosion_merging) ────────────────────────────
FFT_SIZE       = 4096
N_STACKS       = 16
OVERLAP        = 0.25
HOP            = int(FFT_SIZE * (1 - OVERLAP))
SAMPLES_PER_IMG = N_STACKS * HOP + (FFT_SIZE - HOP)
WIN            = get_window('hann', FFT_SIZE).astype(np.float32)

TRAIN_FRAC = 0.60
VAL_FRAC   = 0.20
# TEST_FRAC  = 0.20  (remainder)
N_WORKERS  = 8

print(f'SAMPLES_PER_IMG: {SAMPLES_PER_IMG:,}')
for i, tc in enumerate(TARGET_CLASSES):
    p = BASE_DIR / tc['path_sub']
    iq = np.memmap(str(p), dtype='complex64', mode='r')
    n_imgs = len(iq) // SAMPLES_PER_IMG
    print(f'  label={i}  {tc["name"]:14s}  samples={len(iq):,}  images={n_imgs:,}')

SAMPLES_PER_IMG: 50,176
  label=0  Stack 1.0g      samples=425,696,180  images=8,484
  label=1  Stack 1.5g      samples=468,647,936  images=9,340
  label=2  Stack 2.0g      samples=459,530,344  images=9,158
  label=3  Stack 2.5g      samples=468,615,168  images=9,339


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

def extract_one(iq: np.memmap, img_idx: int) -> np.ndarray:
    """Extract 1040-d feature vector from one image index in an IQ file."""
    start = img_idx * SAMPLES_PER_IMG
    end   = start + SAMPLES_PER_IMG
    if end > len(iq):
        return None

    frames = np.empty((N_STACKS, FFT_SIZE), dtype='complex64')
    for i in range(N_STACKS):
        s = start + i * HOP
        frames[i] = iq[s : s + FFT_SIZE]

    spec    = np.fft.fftshift(np.fft.fft(frames * WIN, axis=1), axes=1)
    mag     = np.abs(spec).astype(np.float32)

    psd_mean = mag.mean(axis=0)
    psd_db   = 20.0 * np.log10(psd_mean + 1e-12)
    psd_512  = psd_db.reshape(512, 8).mean(axis=1)

    psd_var  = mag.var(axis=0)
    var_512  = np.log1p(psd_var.reshape(512, 8).mean(axis=1))

    total_power = float(psd_mean.sum())
    freqs       = np.arange(FFT_SIZE, dtype=np.float32)
    centroid    = float((freqs * psd_mean).sum() / (psd_mean.sum() + 1e-12))
    bandwidth   = float(np.sqrt(((freqs - centroid)**2 * psd_mean).sum() / (psd_mean.sum() + 1e-12)))
    log_psd     = np.log(psd_mean + 1e-12)
    flatness    = float(np.exp(log_psd.mean()) / (psd_mean.mean() + 1e-12))
    p           = psd_mean / (psd_mean.sum() + 1e-12)
    entropy     = float(-np.sum(p * np.log(p + 1e-12)))

    frame_power = mag.sum(axis=1)
    power_var   = float(frame_power.var())
    power_skew  = float(skew(frame_power))
    power_kurt  = float(scipy_kurtosis(frame_power))

    stats = np.array([
        total_power, centroid, bandwidth, flatness,
        entropy, power_var, power_skew, power_kurt
    ], dtype=np.float32)

    phase           = np.angle(spec).astype(np.float32)
    phase_mean_std  = float(np.std(phase.mean(axis=0)))
    phase_var_mean  = float(phase.var(axis=1).mean())
    inst_freq       = np.diff(np.unwrap(phase, axis=0), axis=0)
    if_mean         = float(inst_freq.mean())
    if_std          = float(inst_freq.std())
    i_pwr           = (frames.real**2).mean()
    q_pwr           = (frames.imag**2).mean()
    iq_imbalance    = float(i_pwr / (q_pwr + 1e-12))
    iq_corr         = float(np.corrcoef(
        frames.real.flatten()[:2048], frames.imag.flatten()[:2048]
    )[0, 1])
    amp             = np.abs(frames.flatten()[:4096])
    amp_kurt        = float(scipy_kurtosis(amp))
    amp_skew        = float(skew(amp))

    phase_feats = np.array([
        phase_mean_std, phase_var_mean, if_mean, if_std,
        iq_imbalance, iq_corr, amp_kurt, amp_skew
    ], dtype=np.float32)

    return np.concatenate([psd_512, var_512, stats, phase_feats])  # 1040-d


def extract_split(records, iqs, desc='', n_workers=N_WORKERS):
    """records: list of (label, img_idx). Returns (X, y) arrays."""
    n = len(records)
    feats  = [None] * n
    labels = [None] * n
    t0 = time.time()

    def _worker(i, label, img_idx):
        return i, extract_one(iqs[label], img_idx), label

    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_worker, i, lbl, idx): i
                for i, (lbl, idx) in enumerate(records)}
        done = 0
        for fut in as_completed(futs):
            i, f, lbl = fut.result()
            feats[i], labels[i] = f, lbl
            done += 1
            if done % 500 == 0 or done == n:
                elapsed = time.time() - t0
                eta = elapsed / done * (n - done)
                print(f'  [{desc}] {done}/{n}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

    valid = [(f, l) for f, l in zip(feats, labels) if f is not None]
    return (np.stack([v[0] for v in valid]),
            np.array([v[1] for v in valid]))

print('Feature extraction functions ready.')

Feature extraction functions ready.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. BUILD SPLITS FROM RAW IQ FILES
# ═══════════════════════════════════════════════════════════════════════════════

rng = np.random.default_rng(SEED)

iqs = {i: np.memmap(str(BASE_DIR / tc['path_sub']), dtype='complex64', mode='r')
       for i, tc in enumerate(TARGET_CLASSES)}

train_records, val_records, test_records = [], [], []

for label, iq in iqs.items():
    n_imgs = len(iq) // SAMPLES_PER_IMG
    indices = rng.permutation(n_imgs)
    n_train = int(n_imgs * TRAIN_FRAC)
    n_val   = int(n_imgs * VAL_FRAC)
    train_records.extend((label, int(i)) for i in indices[:n_train])
    val_records.extend((label, int(i))   for i in indices[n_train:n_train + n_val])
    test_records.extend((label, int(i))  for i in indices[n_train + n_val:])
    print(f'  label={label} ({TARGET_CLASSES[label]["name"]}): '
          f'total={n_imgs}  train={n_train}  val={n_val}  test={n_imgs - n_train - n_val}')

rng.shuffle(train_records)
rng.shuffle(val_records)
rng.shuffle(test_records)

print(f'\nTotal: train={len(train_records)}  val={len(val_records)}  test={len(test_records)}')

  label=0 (Stack 1.0g): total=8484  train=5090  val=1696  test=1698
  label=1 (Stack 1.5g): total=9340  train=5604  val=1868  test=1868
  label=2 (Stack 2.0g): total=9158  train=5494  val=1831  test=1833
  label=3 (Stack 2.5g): total=9339  train=5603  val=1867  test=1869

Total: train=21791  val=7262  test=7268


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. FEATURE EXTRACTION WITH CACHING
# ═══════════════════════════════════════════════════════════════════════════════

cache_dir = OUT_DIR / 'feature_cache'
cache_dir.mkdir(exist_ok=True)

def load_or_extract(records, split_name):
    cx = cache_dir / f'{split_name}_X.npy'
    cy = cache_dir / f'{split_name}_y.npy'
    if cx.exists() and cy.exists():
        print(f'  [{split_name}] loaded from cache')
        return np.load(cx), np.load(cy)
    print(f'  [{split_name}] extracting {len(records)} samples...')
    X, y = extract_split(records, iqs, desc=split_name)
    np.save(cx, X); np.save(cy, y)
    return X, y

print('Extracting features...')
X_tr, y_tr   = load_or_extract(train_records, 'train')
X_val, y_val = load_or_extract(val_records,   'val')
X_te, y_te   = load_or_extract(test_records,  'test')

# Clean NaNs / Infs
for arr in (X_tr, X_val, X_te):
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

print(f'X_tr: {X_tr.shape}  X_val: {X_val.shape}  X_te: {X_te.shape}')

Extracting features...
  [train] loaded from cache
  [val] loaded from cache
  [test] loaded from cache
X_tr: (21791, 1040)  X_val: (7262, 1040)  X_te: (7268, 1040)


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4. PREPROCESSING — StandardScaler + PCA
# ═══════════════════════════════════════════════════════════════════════════════

scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_te)

pca = PCA(n_components=256, random_state=SEED)
X_tr_pca  = pca.fit_transform(X_tr_s)
X_val_pca = pca.transform(X_val_s)
X_te_pca  = pca.transform(X_te_s)

evr = pca.explained_variance_ratio_.cumsum()
print(f'PCA 256 components explain {evr[-1]*100:.1f}% of variance')

# Tensors
T = lambda a: torch.tensor(a, dtype=torch.float32)
T_X_tr,  T_y_tr  = T(X_tr_pca),  torch.tensor(y_tr,  dtype=torch.long)
T_X_val, T_y_val = T(X_val_pca), torch.tensor(y_val, dtype=torch.long)
T_X_te,  T_y_te  = T(X_te_pca),  torch.tensor(y_te,  dtype=torch.long)

# Class weights
counts = Counter(y_tr.tolist())
cw = torch.tensor([1.0 / counts[c] for c in range(N_CLASSES)], dtype=torch.float32, device=DEVICE)
cw = cw / cw.sum() * N_CLASSES
print('Class weights:', cw.tolist())

PCA 256 components explain 43.6% of variance
Class weights: [1.0686081647872925, 0.9705951809883118, 0.9900283217430115, 0.9707684516906738]


In [7]:
# 5. MODEL ARCHITECTURE

def _make_device(n_qubits):
    if torch.cuda.is_available():
        try:
            return qml.device('lightning.gpu', wires=n_qubits)
        except Exception:
            pass
    try:
        return qml.device('lightning.qubit', wires=n_qubits)
    except Exception:
        return qml.device('default.qubit', wires=n_qubits)

_test_dev = _make_device(4)
print(f'Quantum device: {_test_dev.name}')


class PriorGuidedQuantumFiLM(nn.Module):
    def __init__(self, c_dim=64, q_dim=4):
        super().__init__()
        self.film_net = nn.Sequential(
            nn.Linear(q_dim, 32), nn.SiLU(), nn.LayerNorm(32),
            nn.Linear(32, c_dim * 3)
        )
        self.gate_logits = nn.Sequential(
            nn.Linear(c_dim + q_dim, c_dim), nn.LayerNorm(c_dim)
        )
        self.q_proj = nn.Sequential(nn.Linear(q_dim, 16), nn.SiLU())
        self.classifier = nn.Sequential(
            nn.Linear(c_dim + 16, 64), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(64, N_CLASSES)
        )
        self.regressor = nn.Sequential(
            nn.Linear(c_dim + 16, 32), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, c, q):
        film_params        = self.film_net(q)
        gamma, beta, alpha = torch.chunk(film_params, 3, dim=1)
        c_mod    = c * (gamma + 1.0) + beta
        raw_gate = self.gate_logits(torch.cat([c_mod, q], dim=1))
        gate_val = torch.sigmoid(raw_gate + alpha)
        fused_c  = gate_val * c_mod + (1 - gate_val) * c
        q_res    = self.q_proj(q)
        feat     = torch.cat([fused_c, q_res], dim=1)
        return self.classifier(feat), self.regressor(feat)


class DenseHybridQNN(nn.Module):
    def __init__(self, n_qubits, n_layers, ansatz, angle_scaling, dropout_c):
        super().__init__()
        self.angle_scaling = angle_scaling
        self.classical_backbone = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_c),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout_c)
        )
        self.qnn_proj = nn.Sequential(nn.Linear(256, n_qubits), nn.Sigmoid())
        dev = _make_device(n_qubits)
        @qml.qnode(dev, interface='torch', diff_method='adjoint')
        def _qnode(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits))
            if ansatz == 'StronglyEntangling':
                qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            else:
                qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        w_shape = ({'weights': (n_layers, n_qubits, 3)}
                   if ansatz == 'StronglyEntangling'
                   else {'weights': (n_layers, n_qubits)})
        self.qnn    = qml.qnn.TorchLayer(_qnode, w_shape)
        self.fusion = PriorGuidedQuantumFiLM(c_dim=64, q_dim=n_qubits)

    def forward(self, x):
        c = self.classical_backbone(x)
        q = self.qnn(self.qnn_proj(x) * self.angle_scaling)
        logits, grams = self.fusion(c, q)
        return logits, grams.squeeze(1)

print('Model classes defined.')


Quantum device: lightning.gpu
Model classes defined.


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6. TRAINING UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def train_epoch(model, loader, opt, clf_loss_fn, reg_loss_fn, alpha=0.3):
    model.train()
    total_loss = correct = n = 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        g_b = torch.tensor(
            [GRAM_VALUES[int(l)] for l in y_b.cpu()],
            dtype=torch.float32, device=DEVICE
        )
        opt.zero_grad()
        logits, grams = model(X_b)
        loss = (1 - alpha) * clf_loss_fn(logits, y_b) + alpha * reg_loss_fn(grams, g_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        n          += len(y_b)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues, gram_preds, gram_trues = [], [], [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        logits, grams = model(X_b)
        preds.extend(logits.argmax(1).cpu().tolist())
        trues.extend(y_b.cpu().tolist())
        gram_preds.extend(grams.cpu().tolist())
        gram_trues.extend([GRAM_VALUES[int(l)] for l in y_b.cpu()])
    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro')
    mae = mean_absolute_error(gram_trues, gram_preds)
    r2  = r2_score(gram_trues, gram_preds)
    return acc, f1, mae, r2, preds, trues

print('Training utilities ready.')

Training utilities ready.


In [ ]:
# 7. PREDEFINED HYPERPARAMETERS  (replaces Q-NAS search)
# Selected based on typical hybrid QNN behaviour on spectral IQ features:
#   - 4 qubits: enough expressiveness, fast adjoint diff on lightning.gpu
#   - StronglyEntangling + 2 layers: captures qubit entanglement without overfit
#   - pi angle scaling: spans full Bloch sphere without wrap-around redundancy
#   - lr=8e-4, wd=1e-4: conservative AdamW defaults for small quantum models
#   - dropout_c=0.2: light regularisation given class balance is already near-uniform

ANGLE_MAP = {'pi_half': np.pi / 2, 'pi': np.pi, 'two_pi': 2 * np.pi}

best_params = {
    'n_qubits':     4,
    'n_layers':     2,
    'ansatz':       'StronglyEntangling',
    'angle_scaling': 'pi',
    'lr':            8e-4,
    'weight_decay':  1e-4,
    'dropout_c':     0.2,
    'batch_size':    64,
}

print('Using predefined hyperparameters:')
for k, v in best_params.items():
    print(f'  {k}: {v}')


In [ ]:
# Optuna importance plot
try:
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'qnas_importances.png', dpi=200)
    plt.show()
except Exception as e:
    print(f'Importance plot skipped: {e}')

In [ ]:
# 8. FINAL MODEL TRAINING WITH BEST PARAMS
# Two-phase strategy to work around quantum circuit simulation being sequential:
#   Phase 1 (WARMUP_EPOCHS): train all params including QNN — slow but needed
#                            to learn good quantum weights.
#   Phase 2 (remaining):     freeze QNN, train classical layers only — runs at
#                            full GPU speed since no quantum circuit calls during
#                            backward pass.
from tqdm.auto import tqdm as tqdm_nb

WARMUP_EPOCHS = 10   # epochs with QNN unfrozen
FINAL_EPOCHS  = 100
PATIENCE      = 20

best_bs = best_params['batch_size']
tr_loader  = DataLoader(TensorDataset(T_X_tr,  T_y_tr),  batch_size=best_bs, shuffle=True,  drop_last=True)
val_loader = DataLoader(TensorDataset(T_X_val, T_y_val), batch_size=best_bs, shuffle=False)
te_loader  = DataLoader(TensorDataset(T_X_te,  T_y_te),  batch_size=best_bs, shuffle=False)

final_model = DenseHybridQNN(
    n_qubits      = best_params['n_qubits'],
    n_layers      = best_params['n_layers'],
    ansatz        = best_params['ansatz'],
    angle_scaling = ANGLE_MAP[best_params['angle_scaling']],
    dropout_c     = best_params['dropout_c']
).to(DEVICE)

clf_loss  = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
reg_loss  = nn.HuberLoss(delta=0.5)

def make_optimizer(model, lr, wd):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=wd
    )

opt = make_optimizer(final_model, best_params['lr'], best_params['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)

best_val_f1  = 0.0
best_weights = None
no_improve   = 0
history      = {'tr_acc': [], 'val_f1': [], 'val_mae': [], 'val_r2': [], 'phase': []}
phase        = 1

epoch_bar = tqdm_nb(range(1, FINAL_EPOCHS + 1), desc='Training', unit='epoch')
for ep in epoch_bar:

    # Switch to phase 2: freeze QNN, rebuild optimizer over classical params only
    if ep == WARMUP_EPOCHS + 1 and phase == 1:
        for p in final_model.qnn.parameters():
            p.requires_grad_(False)
        opt = make_optimizer(final_model, best_params['lr'], best_params['weight_decay'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
        phase = 2
        epoch_bar.write(f'[ep {ep}] QNN frozen — switching to classical-only training')

    tr_loss, tr_acc                        = train_epoch(final_model, tr_loader, opt, clf_loss, reg_loss)
    val_acc, val_f1, val_mae, val_r2, _, _ = evaluate(final_model, val_loader)
    scheduler.step(val_f1)

    history['tr_acc'].append(tr_acc)
    history['val_f1'].append(val_f1)
    history['val_mae'].append(val_mae)
    history['val_r2'].append(val_r2)
    history['phase'].append(phase)

    epoch_bar.set_postfix({
        'ph':     phase,
        'tr_acc': f'{tr_acc:.3f}',
        'val_f1': f'{val_f1:.3f}',
        'mae':    f'{val_mae:.3f}',
        'best':   f'{best_val_f1:.3f}',
        'wait':   f'{no_improve}/{PATIENCE}'
    })

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        no_improve   = 0
        best_weights = copy.deepcopy(final_model.state_dict())
        torch.save(best_weights, OUT_DIR / 'stack_qnn_best.pt')
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        epoch_bar.write(f'Early stopping at epoch {ep}')
        break

print(f'Best validation F1: {best_val_f1:.4f}')


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['tr_acc'], label='Train Acc')
axes[0].plot(history['val_f1'], label='Val F1')
axes[0].set_title('Accuracy & F1 vs Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['val_mae'], color='orange')
axes[1].set_title('Val MAE (g) vs Epoch'); axes[1].set_ylabel('MAE'); axes[1].grid(True)

axes[2].plot(history['val_r2'], color='green')
axes[2].set_title('Val R² vs Epoch'); axes[2].set_ylabel('R²'); axes[2].grid(True)

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=200)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9. TEST SET EVALUATION
# ═══════════════════════════════════════════════════════════════════════════════

final_model.load_state_dict(best_weights)
te_acc, te_f1, te_mae, te_r2, te_pred, te_true = evaluate(final_model, te_loader)

print('=' * 60)
print('FINAL EVALUATION — UNSEEN TEST SET')
print('=' * 60)
print(f'  Accuracy :  {te_acc:.4f}')
print(f'  F1-macro :  {te_f1:.4f}')
print(f'  MAE (g)  :  {te_mae:.4f}')
print(f'  R²       :  {te_r2:.4f}')
print()
print('Classification Report:')
print(classification_report(te_true, te_pred, target_names=LABEL_NAMES, digits=4))
print('Confusion Matrix:')
cm = confusion_matrix(te_true, te_pred)
print(cm)

In [ ]:
# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(LABEL_NAMES, rotation=30, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Stack Dataset')
plt.colorbar(im, ax=ax)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig(OUT_DIR / 'confusion_matrix.png', dpi=200)
plt.show()

# Save summary
summary = {
    'best_params': best_params,
    'best_val_f1': float(best_val_f1),
    'test': {'acc': float(te_acc), 'f1': float(te_f1), 'mae': float(te_mae), 'r2': float(te_r2)}
}
with open(OUT_DIR / 'results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved to {OUT_DIR}')